# Voice Shield · 02 · Fine-tune for Hindi, English, real mics and phone calls

Starts from the public XLS-R + SLS checkpoint and fine-tunes it so it:
- stops calling real human voices "AI" when they come from ordinary mics, phones or quiet rooms
- works on **Hindi** as well as English
- works on **phone-quality audio** (8 kHz, μ-law, phone codecs)
- does not use volume, silence, noise or room echo as a shortcut

It prints a **before vs after** table at the end.

**AI voices used for training**
- English: ASVspoof 2019 (older tools) + **MLAAD**: 130+ recent tools, including ElevenLabs, MiniMax, F5-TTS,
  XTTS-v2, Chatterbox, VibeVoice, Qwen3-TTS, Fish, Kokoro and many more.
- Hindi: IndicSynth (FreeVC, XTTS-v2) + MLAAD Hindi (IndicF5, XTTS-v2, VoxCPM2, Indri and more).

**Newest tools kept out of training**, so the "newest AI tools never trained on" rows show how the model
does on a tool it has never seen: ElevenLabs v3, OpenAI TTS-1 HD, Cartesia Sonic-3, Gemini TTS,
MiniMax 2.8, Resemble.ai, Inworld TTS-2, Chatterbox-Turbo (English) and Veena, Ringg (Hindi).
Older versions from several of the same companies are trained on. Empty the holdout lists for the final model.

**Real voices used for training**: ASVspoof 2019, FLEURS (Hindi + English), LibriSpeech (English audiobooks,
to match MLAAD's audiobook-style fakes), **Svarah (Indian-accented English)** and Kathbath (Hindi).

**Splits**: train / dev / test, never sharing a speaker where the dataset says who is speaking.
Dev picks the best checkpoint. Test is only reported. In-the-Wild and the held-out tools are test only.

**Before you press Run All**
1. Settings → Accelerator → **GPU P100** or **GPU T4 x2** (only one GPU is used).
2. Settings → **Internet on**.
3. Add Input → search **"ASVspoof 2019"** and add one that contains `ASVspoof2019.LA.cm.train.trn.txt`.
4. Add Input → your **In-the-Wild** dataset from notebook 01 (optional, but gives the real-world test row).
5. **Strongly recommended: HuggingFace token.** Without it MLAAD (newest AI voices), Svarah and Kathbath are skipped.
   - log in to huggingface.co and accept the access terms on all three:
     **huggingface.co/datasets/mueller91/MLAAD**, **huggingface.co/datasets/ai4bharat/Svarah**,
     **huggingface.co/datasets/ai4bharat/Kathbath**
   - Settings → Access Tokens → new **Read** token
   - Kaggle: Add-ons → Secrets → add **HF_TOKEN** with that token, and switch it on for this notebook
6. Optional, our own recordings: upload them as a Kaggle dataset with a `vs_labels.csv` file
   (columns `file,label,language,speaker,split,tool`) and add it as an input. People marked `train` are
   trained on; people marked `test` become the "Our recordings" test row.

**Run A** (`RUN = "A"`, about 2–3 hours on a T4): proves generalization, saves `voice_shield_ft.pth`.
**Run B** (`RUN = "B"`, about 1–1.5 hours): attach run A's notebook output as an input, trains on every
tool starting from run A's model, saves `voice_shield_final.pth` (the demo model).
Use Save Version → Save & Run All so it keeps running with the browser closed, then download the `.pth`
file from the version's Output tab.

## Our code
Run these cells as they are. Each one writes one file of our `vs` package.

In [ ]:
import os
os.makedirs('vs', exist_ok=True)

In [ ]:
%%writefile vs/__init__.py
"""Voice Shield: AI-voice and scam-call detection."""


In [ ]:
%%writefile vs/audio.py
"""Audio helpers: loading, clean-up, and phone-call simulation.

Everything works on mono float32 numpy arrays in the range [-1, 1].
The detector expects 16 kHz audio; phone audio (Twilio) is 8 kHz mu-law.
"""

from __future__ import annotations

import shutil
import subprocess
import tempfile
from math import gcd
from pathlib import Path

import numpy as np
import soundfile as sf
from scipy.signal import butter, resample_poly, sosfilt

SR = 16000            # detector sample rate
PHONE_SR = 8000       # telephone sample rate
WINDOW = 64600        # samples the SLS model was trained on (~4.04 s at 16 kHz)


# ---------------------------------------------------------------- loading

def resample(x: np.ndarray, sr_in: int, sr_out: int) -> np.ndarray:
    if sr_in == sr_out:
        return x.astype(np.float32)
    g = gcd(sr_in, sr_out)
    return resample_poly(x, sr_out // g, sr_in // g).astype(np.float32)


def load_audio(path: str | Path, sr: int = SR) -> np.ndarray:
    """Load any file libsndfile can read (wav, flac, ogg, mp3) as mono at `sr`."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    return resample(x.mean(axis=1), file_sr, sr)


def to_mono_float(raw: bytes, sr_in: int, sr: int = SR) -> np.ndarray:
    """Decode an uploaded file's bytes."""
    with tempfile.NamedTemporaryFile(suffix=".audio", delete=False) as f:
        f.write(raw)
        tmp = f.name
    try:
        return load_audio(tmp, sr)
    finally:
        Path(tmp).unlink(missing_ok=True)


# ---------------------------------------------------------------- clean-up

def normalize_loudness(x: np.ndarray, target_dbfs: float = -23.0) -> np.ndarray:
    """Scale to a fixed RMS level so loudness can't be used as a shortcut."""
    rms = float(np.sqrt(np.mean(x ** 2)) + 1e-9)
    y = x * (10 ** (target_dbfs / 20) / rms)
    return np.clip(y, -1.0, 1.0).astype(np.float32)


def trim_silence(x: np.ndarray, sr: int = SR, top_db: float = 35.0,
                 frame_ms: float = 25.0) -> np.ndarray:
    """Remove silence at the start and end (energy based).

    Silence length alone predicts real vs fake in ASVspoof 2019, so we cut it.
    """
    frame = int(sr * frame_ms / 1000)
    if len(x) < frame * 2:
        return x
    n = len(x) // frame
    energy = np.array([np.mean(x[i * frame:(i + 1) * frame] ** 2) for i in range(n)])
    db = 10 * np.log10(energy + 1e-12)
    keep = np.where(db > db.max() - top_db)[0]
    if len(keep) == 0:
        return x
    return x[keep[0] * frame:(keep[-1] + 1) * frame]


def repeat_pad(x: np.ndarray, length: int = WINDOW) -> np.ndarray:
    """Cut or repeat-pad to `length` samples (same as the SLS training code)."""
    if len(x) >= length:
        return x[:length]
    reps = length // max(len(x), 1) + 1
    return np.tile(x, reps)[:length]


def windows(x: np.ndarray, length: int = WINDOW, hop: int = SR) -> list[np.ndarray]:
    """Split a long clip into overlapping model-sized windows."""
    if len(x) <= length:
        return [repeat_pad(x, length)]
    starts = list(range(0, len(x) - length + 1, hop))
    if starts[-1] + length < len(x):
        starts.append(len(x) - length)
    return [x[s:s + length] for s in starts]


# ---------------------------------------------------------------- G.711 mu-law
# Twilio Media Streams always send 8 kHz G.711 mu-law, mono.

def _build_g711_table() -> np.ndarray:
    table = np.zeros(256, dtype=np.float32)
    for i in range(256):
        u = ~i & 0xFF
        sign = u & 0x80
        exponent = (u >> 4) & 0x07
        mantissa = u & 0x0F
        sample = ((mantissa << 3) + 0x84) << exponent
        sample -= 0x84
        table[i] = (-sample if sign else sample) / 32768.0
    return table


_G711 = _build_g711_table()


def g711_decode(b: bytes) -> np.ndarray:
    return _G711[np.frombuffer(b, dtype=np.uint8)]


def g711_encode(x: np.ndarray) -> bytes:
    pcm = np.clip(x * 32768.0, -32635, 32635).astype(np.int32)
    sign = np.where(pcm < 0, 0x80, 0)
    mag = np.abs(pcm) + 0x84
    exponent = np.clip(np.floor(np.log2(mag)).astype(np.int32) - 7, 0, 7)
    mantissa = (mag >> (exponent + 3)) & 0x0F
    return (~(sign | (exponent << 4) | mantissa) & 0xFF).astype(np.uint8).tobytes()


# ---------------------------------------------------------------- phone simulation

_BANDPASS = butter(4, [300, 3400], btype="bandpass", fs=PHONE_SR, output="sos")


def _ffmpeg_codec(x8k: np.ndarray, codec: str) -> np.ndarray | None:
    """Round-trip 8 kHz audio through a real phone codec using ffmpeg, if present."""
    exe = shutil.which("ffmpeg")
    if exe is None:
        return None
    fmt = {"amr": ("libopencore_amrnb", "amr", ["-b:a", "12.2k"]),
           "gsm": ("libgsm", "gsm", [])}[codec]
    with tempfile.TemporaryDirectory() as d:
        src, enc, out = Path(d, "in.wav"), Path(d, f"enc.{fmt[1]}"), Path(d, "out.wav")
        sf.write(src, x8k, PHONE_SR)
        ok = subprocess.run([exe, "-y", "-loglevel", "error", "-i", str(src), "-ar", "8000",
                             "-ac", "1", "-c:a", fmt[0], *fmt[2], str(enc)],
                            capture_output=True).returncode == 0
        if not ok:
            return None
        subprocess.run([exe, "-y", "-loglevel", "error", "-i", str(enc), "-ar", "8000",
                        str(out)], capture_output=True)
        y, _ = sf.read(out, dtype="float32")
        return y[:len(x8k)]


def phone_simulate(x: np.ndarray, sr: int = SR, codec: str | None = None,
                   noise_snr_db: float | None = None,
                   rng: np.random.Generator | None = None,
                   bandpass: bool = True) -> np.ndarray:
    """Make clean audio sound like a phone call, then return it at `sr`.

    Steps: downsample to 8 kHz, telephone band (300-3400 Hz), optional real codec
    (AMR-NB or GSM via ffmpeg), optional background noise, G.711 mu-law (what
    Twilio delivers), back up to `sr`.
    """
    rng = rng or np.random.default_rng()
    y = resample(x, sr, PHONE_SR)
    if bandpass:
        y = sosfilt(_BANDPASS, y).astype(np.float32)
    if codec:
        coded = _ffmpeg_codec(y, codec)
        if coded is not None:
            y = coded
    if noise_snr_db is not None:
        p = np.mean(y ** 2) + 1e-12
        y = y + rng.normal(0, np.sqrt(p / 10 ** (noise_snr_db / 10)), len(y)).astype(np.float32)
    y = g711_decode(g711_encode(np.clip(y, -1, 1)))
    return resample(y, PHONE_SR, sr)


def has_ffmpeg_codecs() -> bool:
    return shutil.which("ffmpeg") is not None


In [ ]:
%%writefile vs/metrics.py
"""Equal Error Rate and related numbers for the results table."""

from __future__ import annotations

import numpy as np


def compute_eer(p_fake_real: np.ndarray, p_fake_fake: np.ndarray) -> tuple[float, float]:
    """EER and the threshold where it happens.

    p_fake_real: fake-probabilities given to real (bonafide) clips
    p_fake_fake: fake-probabilities given to AI (spoof) clips
    Returns (eer as a fraction, threshold on p_fake).
    """
    scores = np.concatenate([p_fake_real, p_fake_fake])
    labels = np.concatenate([np.zeros(len(p_fake_real)), np.ones(len(p_fake_fake))])
    order = np.argsort(scores)
    scores, labels = scores[order], labels[order]
    n_real, n_fake = len(p_fake_real), len(p_fake_fake)
    # threshold just above scores[i]: everything <= scores[i] is called real
    false_alarm = 1 - np.cumsum(labels == 0) / n_real     # real clips called fake
    miss = np.cumsum(labels == 1) / n_fake                # fake clips called real
    i = int(np.argmin(np.abs(false_alarm - miss)))
    return float((false_alarm[i] + miss[i]) / 2), float(scores[i])


def rates_at(p_fake_real: np.ndarray, p_fake_fake: np.ndarray, threshold: float) -> dict:
    """False-alarm and miss rates if we call 'fake' when p_fake > threshold."""
    return {
        "false_alarm_on_real": float(np.mean(p_fake_real > threshold)),
        "missed_fakes": float(np.mean(p_fake_fake <= threshold)),
    }


In [ ]:
%%writefile vs/model.py
"""XLS-R + SLS deepfake detector, rebuilt on HuggingFace transformers.

The public checkpoint (Zhang et al. 2024, reproduced by Sukhdeve) was trained
with fairseq, which is hard to install on Windows and new Pythons. This file
rebuilds the same network with `transformers.Wav2Vec2Model` and converts the
fairseq weights on load, so the laptop and Kaggle run the same code.

Model input: (batch, 64600) raw 16 kHz waveform, no normalisation.
Model output: log-probabilities for [spoof, bonafide]  (index 1 = real voice).
"""

from __future__ import annotations

import re

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Wav2Vec2Config, Wav2Vec2Model

XLSR_NAME = "facebook/wav2vec2-xls-r-300m"
CKPT_REPO = "sukhdeveyash/XLS-R-SLS-Deepfake-Detection"
CKPT_FILE = "v1/epoch_2.pth"   # the repo's recommended checkpoint (7.84% EER on In-the-Wild)


def xlsr_config() -> Wav2Vec2Config:
    cfg = Wav2Vec2Config.from_pretrained(XLSR_NAME)
    # fairseq was called with mask=False and every layer must run, so turn off
    # SpecAugment masking and LayerDrop.
    cfg.apply_spec_augment = False
    cfg.mask_time_prob = 0.0
    cfg.mask_feature_prob = 0.0
    cfg.layerdrop = 0.0
    return cfg


class SLSModel(nn.Module):
    """XLS-R 300M -> Selective Layer Summarization -> small classifier head."""

    def __init__(self, cfg: Wav2Vec2Config | None = None):
        super().__init__()
        self.ssl = Wav2Vec2Model(cfg or xlsr_config())
        self.first_bn = nn.BatchNorm2d(num_features=1)
        self.selu = nn.SELU(inplace=True)
        self.fc0 = nn.Linear(1024, 1)
        self.fc1 = nn.Linear(22847, 1024)   # 341 * 67 after 3x3 max-pool of (201 frames, 1024 dims)
        self.fc3 = nn.Linear(1024, 2)

        # fairseq's layer_results are the raw outputs of all 24 transformer
        # layers. HF applies a final LayerNorm to its last hidden state, so we
        # hook the layers directly to get exactly the same tensors.
        self._layer_out: list[torch.Tensor] = []
        for layer in self.ssl.encoder.layers:
            layer.register_forward_hook(self._grab)

    def _grab(self, _module, _inp, out):
        self._layer_out.append(out[0] if isinstance(out, tuple) else out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self._layer_out = []
        self.ssl(x)
        layers = self._layer_out                       # 24 x (B, T, 1024)
        self._layer_out = []

        pooled = torch.stack([h.mean(dim=1) for h in layers], dim=1)   # (B, 24, 1024)
        full = torch.stack(layers, dim=1)                              # (B, 24, T, 1024)
        weight = torch.sigmoid(self.fc0(pooled)).unsqueeze(-1)          # (B, 24, 1, 1)
        feat = (full * weight).sum(dim=1).unsqueeze(1)                 # (B, 1, T, 1024)

        x = self.selu(self.first_bn(feat))
        x = F.max_pool2d(x, (3, 3))
        x = torch.flatten(x, 1)
        x = self.selu(self.fc1(x))
        x = self.selu(self.fc3(x))
        return F.log_softmax(x, dim=1)


# ---------------------------------------------------------------- fairseq -> HF weights

_RULES = [
    (r"feature_extractor\.conv_layers\.(\d+)\.0\.", r"feature_extractor.conv_layers.\1.conv."),
    (r"feature_extractor\.conv_layers\.(\d+)\.2\.1\.", r"feature_extractor.conv_layers.\1.layer_norm."),
    (r"^layer_norm\.", "feature_projection.layer_norm."),
    (r"^post_extract_proj\.", "feature_projection.projection."),
    (r"encoder\.layers\.(\d+)\.self_attn\.", r"encoder.layers.\1.attention."),
    (r"encoder\.layers\.(\d+)\.self_attn_layer_norm\.", r"encoder.layers.\1.layer_norm."),
    (r"encoder\.layers\.(\d+)\.fc1\.", r"encoder.layers.\1.feed_forward.intermediate_dense."),
    (r"encoder\.layers\.(\d+)\.fc2\.", r"encoder.layers.\1.feed_forward.output_dense."),
]
# pretraining-only parts; mask_emb is unused because masking is off
_SKIP = ("quantizer.", "project_q.", "final_proj.", "label_embs", "target_glu.", "mask_emb")


def _pos_conv_keys(hf_keys: set[str]) -> tuple[str, str]:
    base = "ssl.encoder.pos_conv_embed.conv."
    if base + "parametrizations.weight.original0" in hf_keys:
        return base + "parametrizations.weight.original0", base + "parametrizations.weight.original1"
    return base + "weight_g", base + "weight_v"


def convert_fairseq_checkpoint(sd: dict, model: SLSModel) -> dict:
    """Rename a fairseq SLS state dict so it loads into SLSModel."""
    hf_keys = set(model.state_dict().keys())
    g_key, v_key = _pos_conv_keys(hf_keys)
    out = {}
    for k, v in sd.items():
        k = k.removeprefix("module.")
        if not k.startswith("ssl_model.model."):
            out[k] = v                          # classifier head: same names
            continue
        k = k.removeprefix("ssl_model.model.")
        if k.startswith(_SKIP):
            continue
        if k.startswith("encoder.pos_conv.0."):
            tail = k.removeprefix("encoder.pos_conv.0.")
            target = {"weight_g": g_key, "weight_v": v_key,
                      "parametrizations.weight.original0": g_key,
                      "parametrizations.weight.original1": v_key,
                      "bias": "ssl.encoder.pos_conv_embed.conv.bias"}[tail]
            out[target] = v
            continue
        for pat, rep in _RULES:
            k = re.sub(pat, rep, k)
        out["ssl." + k] = v
    return out


def load_sls(checkpoint_path: str, device: str = "cpu") -> SLSModel:
    model = SLSModel()
    raw = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    if isinstance(raw, dict) and "state_dict" in raw:
        raw = raw["state_dict"]
    sd = convert_fairseq_checkpoint(raw, model) if any(
        k.removeprefix("module.").startswith("ssl_model.") for k in raw) else raw
    missing, unexpected = model.load_state_dict(sd, strict=False)
    missing = [k for k in missing if not k.endswith("num_batches_tracked")]
    if missing or unexpected:
        raise RuntimeError(f"Checkpoint did not match the model.\n"
                           f"Missing ({len(missing)}): {missing[:10]}\n"
                           f"Unexpected ({len(unexpected)}): {unexpected[:10]}")
    return model.to(device).eval()


def download_checkpoint(filename: str = CKPT_FILE, cache_dir: str | None = None) -> str:
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=CKPT_REPO, filename=filename, cache_dir=cache_dir)


In [ ]:
%%writefile vs/augment.py
"""Training-time augmentation.

Goal: the model must not be able to use recording conditions (volume,
silence, microphone, phone line, room, noise) to tell real from fake. Every
augmentation is applied at random to real and fake clips alike, so these
conditions carry no label information.

The last two steps always run, because the detector does the same at
inference: trim silence at the ends, and set the window to -23 dBFS.
"""

from __future__ import annotations

import numpy as np
from scipy.signal import fftconvolve

from . import audio


def _pink_noise(n: int, rng: np.random.Generator) -> np.ndarray:
    spec = np.fft.rfft(rng.normal(size=n))
    f = np.arange(len(spec))
    spec /= np.sqrt(np.maximum(f, 1))
    y = np.fft.irfft(spec, n)
    return (y / (np.std(y) + 1e-9)).astype(np.float32)


def add_noise(x: np.ndarray, rng: np.random.Generator, snr_db: tuple = (8, 30)) -> np.ndarray:
    noise = _pink_noise(len(x), rng) if rng.random() < 0.5 else rng.normal(size=len(x)).astype(np.float32)
    p = np.mean(x ** 2) + 1e-12
    snr = rng.uniform(*snr_db)
    return x + noise * np.sqrt(p / 10 ** (snr / 10))


def add_reverb(x: np.ndarray, rng: np.random.Generator, sr: int = audio.SR) -> np.ndarray:
    """Synthetic room: exponentially decaying noise as the impulse response."""
    rt60 = rng.uniform(0.15, 0.7)
    n = int(sr * rt60)
    t = np.arange(n) / sr
    rir = rng.normal(size=n) * np.exp(-6.9 * t / rt60)
    rir[0] = 1.0
    rir /= np.sqrt(np.sum(rir ** 2))
    wet = fftconvolve(x, rir)[:len(x)]
    mix = rng.uniform(0.3, 1.0)
    return ((1 - mix) * x + mix * wet).astype(np.float32)


def random_crop(x: np.ndarray, rng: np.random.Generator, length: int = audio.WINDOW) -> np.ndarray:
    if len(x) <= length:
        return audio.repeat_pad(x, length)
    s = rng.integers(0, len(x) - length + 1)
    return x[s:s + length]


def augment(x: np.ndarray, rng: np.random.Generator, p_phone: float = 0.5,
            p_noise: float = 0.3, p_reverb: float = 0.2, p_clip: float = 0.1) -> np.ndarray:
    """One random training view of a 16 kHz clip, ready for the model."""
    x = audio.trim_silence(x)
    # take a bit more than one window so later steps have context, then crop
    x = random_crop(x, rng, min(len(x), audio.WINDOW + audio.SR))
    x = audio.normalize_loudness(x, rng.uniform(-40, -12))     # random input volume
    if rng.random() < p_reverb:
        x = add_reverb(x, rng)
    if rng.random() < p_noise:
        x = add_noise(x, rng)
    if rng.random() < p_clip:
        x = np.clip(x * rng.uniform(2, 6), -1, 1)               # overdriven mic
    if rng.random() < p_phone:
        x = audio.phone_simulate(x, rng=rng, bandpass=rng.random() < 0.6,
                                 codec=rng.choice([None, None, "amr", "gsm"]),
                                 noise_snr_db=rng.uniform(20, 40) if rng.random() < 0.3 else None)
    x = random_crop(x, rng)
    return audio.normalize_loudness(x)


def eval_view(x: np.ndarray, phone: bool, seed: int) -> np.ndarray:
    """Deterministic view for evaluation, matching what the app does."""
    x = audio.trim_silence(x)
    if phone:
        x = audio.phone_simulate(x, rng=np.random.default_rng(seed))
    return audio.normalize_loudness(audio.repeat_pad(x))


In [ ]:
%%writefile vs/train_data.py
"""Training and evaluation data: download, extract, and list clips.

Every clip becomes one row (a dict) with:
    path, label ('real' | 'fake'), lang ('en' | 'hi'), source, speaker,
    generator (cloning tool, '' for real), split ('train' | 'dev' | 'test')

Splits never share speakers where the source tells us who is speaking, and
IndicSynth's XTTS-v2 clips are kept out of training entirely so we can measure
"a cloning tool the model has never seen".
"""

from __future__ import annotations

import io
import os
import time
import zlib
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
from torch.utils.data import Dataset

from . import audio
from .augment import augment, eval_view


def find_file(name: str, root: str = "/kaggle/input") -> str | None:
    """First file called `name` under root, without walking into audio folders.

    glob('**') would list hundreds of thousands of audio files in ASVspoof.
    """
    skip = {"flac", "wav", "wavs", "audio"}
    for d, dirs, files in os.walk(root):
        for f in files:
            if f.lower() == name.lower():          # some uploads change the case
                return os.path.join(d, f)
        dirs[:] = [x for x in dirs if x.lower() not in skip]
    return None


def _speaker_split(speaker, dev_share: int = 10) -> str:
    """Stable train/dev split by speaker (about 1 in `dev_share` goes to dev)."""
    return "dev" if zlib.crc32(str(speaker).encode()) % dev_share == 0 else "train"


# ---------------------------------------------------------------- parquet extraction

def _audio_bytes(row: dict) -> bytes | None:
    """Encoded audio from a HuggingFace row, whatever the audio column is called."""
    a = row.get("audio")
    if isinstance(a, dict) and a.get("bytes"):
        return a["bytes"]
    for v in row.values():
        if isinstance(v, dict) and v.get("bytes"):
            return v["bytes"]
    return None


def _audio_name(row: dict) -> str:
    for v in row.values():
        if isinstance(v, dict) and v.get("path"):
            return str(v["path"])
    return ""


def extract_parquet(parquet_path: str, out_dir: str, prefix: str, max_clips: int,
                    row_fn, max_seconds: float = 8.0) -> list[dict]:
    """Decode audio from a HuggingFace parquet file into 16 kHz wav files.

    row_fn(row) returns the metadata dict for a row, or None to skip it.
    Rows are taken evenly across the file so we don't only get its first speaker.
    """
    import pyarrow.parquet as pq
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    pf = pq.ParquetFile(parquet_path)
    print(f"  {Path(parquet_path).name}: {pf.metadata.num_rows} rows, columns {pf.schema_arrow.names}")
    step = max(1, pf.metadata.num_rows // max_clips)
    rows, idx = [], 0
    for batch in pf.iter_batches(batch_size=64):
        for row in batch.to_pylist():
            idx += 1
            if (idx - 1) % step or len(rows) >= max_clips:
                continue
            meta = row_fn(row)
            if meta is None:
                continue
            raw = _audio_bytes(row)
            if not raw:
                continue
            try:
                x, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=True)
            except Exception:
                continue
            x = audio.resample(x.mean(axis=1), sr, audio.SR)[: int(max_seconds * audio.SR)]
            if len(x) < audio.SR:                       # skip clips under 1 s
                continue
            path = out / f"{prefix}_{idx:07d}.wav"
            sf.write(path, x, audio.SR, subtype="PCM_16")
            rows.append({**meta, "path": str(path)})
        if len(rows) >= max_clips:
            break
    return rows


def _hf_extract(repo: str, filename: str, out_dir: str, prefix: str, max_clips: int, row_fn,
                token: str | None = None) -> list[dict]:
    """Download one parquet file, extract clips from it, then delete it to save disk."""
    from huggingface_hub import hf_hub_download
    tmp = Path(out_dir) / "_download"
    path = hf_hub_download(repo_id=repo, repo_type="dataset", filename=filename, token=token,
                           local_dir=str(tmp))
    try:
        return extract_parquet(path, out_dir, prefix, max_clips, row_fn)
    finally:
        Path(path).unlink(missing_ok=True)


# ---------------------------------------------------------------- sources

def fleurs(lang: str, split: str, out_dir: str, max_clips: int) -> list[dict]:
    """Real speech from FLEURS. lang: 'hi_in' or 'en_us'. split: train | validation | test."""
    short = lang[:2]
    ours = {"train": "train", "validation": "dev", "test": "test"}[split]
    return _hf_extract("google/fleurs", f"parquet-data/{lang}/{split}-00000-of-00001.parquet",
                       f"{out_dir}/fleurs_{lang}_{split}", f"fleurs_{short}_{split}", max_clips,
                       lambda r: {"label": "real", "lang": short, "source": f"fleurs_{short}",
                                  "speaker": f"fleurs_{short}_{split}", "generator": "",
                                  "split": ours})


def indicsynth_hindi(shards: list[int], out_dir: str, per_shard: int,
                     holdout_xtts: bool = False) -> list[dict]:
    """Hindi AI speech from IndicSynth (107 shards, ~0.5 GB each).

    The Hindi shards are sorted by tool: roughly 0-54 are FreeVC (voice
    conversion) and 56-106 are XTTS-v2 (voice-cloning TTS). With holdout_xtts,
    every XTTS-v2 clip goes to test as an "unseen tool"; otherwise XTTS-v2 is
    trained on too, split by speaker like FreeVC.
    """
    def meta(r):
        gen = r.get("Generative Model") or "unknown"
        spk = r.get("Target Speaker ID")
        split = "test" if holdout_xtts and gen == "xtts_v2" else _speaker_split(spk)
        return {"label": "fake", "lang": "hi", "source": "indicsynth", "speaker": f"is_{spk}",
                "generator": gen, "split": split}
    rows = []
    for s in shards:
        rows += _hf_extract("vdivyasharma/IndicSynth", f"Hindi/train-{s:05d}-of-00107.parquet",
                            f"{out_dir}/indicsynth", f"is{s:03d}", per_shard, meta)
        print(f"  IndicSynth shard {s}: {len(rows)} clips so far")
    return rows


def kathbath_hindi(shards: list[int], out_dir: str, per_shard: int, token: str) -> list[dict]:
    """Real Hindi speech from Kathbath (needs a HuggingFace token; accept the terms on its page first)."""
    def meta(r):
        spk = r.get("speaker_id") or r.get("speaker") or r.get("client_id")
        if spk is None:
            # no speaker column: fall back to the file name so clips still spread over train and dev
            spk = _audio_name(r) or str(zlib.crc32(repr(sorted(k for k in r)).encode() + os.urandom(4)))
        return {"label": "real", "lang": "hi", "source": "kathbath", "speaker": f"kb_{spk}",
                "generator": "", "split": _speaker_split(spk)}
    rows = []
    for s in shards:
        rows += _hf_extract("ai4bharat/Kathbath", f"hindi/train-{s:05d}-of-00032.parquet",
                            f"{out_dir}/kathbath", f"kb{s:02d}", per_shard, meta, token)
    return rows


def svarah(train_shards: list[int], test_shards: list[int], out_dir: str, per_shard: int,
           token: str) -> list[dict]:
    """Real Indian-accented English (AI4Bharat Svarah, needs HF_TOKEN; 3 files of ~365 MB).

    Without it, the only real English is American/British, and real Indians
    speaking English could look "unusual" to the model and be flagged as AI.
    """
    rows = []
    for s in train_shards + test_shards:
        def meta(r, test=s in test_shards):
            spk = r.get("speaker_id") or r.get("speaker") or _audio_name(r).split("_")[0] or None
            if spk is None:
                spk = str(zlib.crc32(os.urandom(4)))
            return {"label": "real", "lang": "en", "source": "svarah", "speaker": f"sv_{spk}",
                    "generator": "", "split": "test" if test else _speaker_split(spk)}
        rows += _hf_extract("ai4bharat/Svarah", f"data/test-{s:05d}-of-00003.parquet",
                            f"{out_dir}/svarah", f"sv{s}", per_shard, meta, token)
    return rows


def librispeech(parts: dict[str, list[int]], out_dir: str, per_shard: int) -> list[dict]:
    """Real English audiobook speech (open, many speakers).

    MLAAD's English fakes are audiobook text read by AI voices, so we need real
    audiobook reading too, or "audiobook style" would look like a sign of fakes.
    parts: e.g. {"train.clean.100": [0, 7], "train.other.500": [0, 32], "test.clean": [0]}
    """
    rows = []
    for part, shards in parts.items():
        for s in shards:
            def meta(r, part=part):
                spk = r.get("speaker_id")
                split = "test" if part.startswith("test") else _speaker_split(spk)
                return {"label": "real", "lang": "en", "source": "librispeech", "speaker": f"ls_{spk}",
                        "generator": "", "split": split}
            rows += _hf_extract("openslr/librispeech_asr", f"all/{part}/{s:04d}.parquet",
                                f"{out_dir}/librispeech", f"ls_{part}_{s:02d}", per_shard, meta)
    return rows


def mlaad(lang: str, out_dir: str, per_tool: int, token: str, holdout: list[str],
          max_tools: int | None = None, workers: int = 8, per_holdout_tool: int | None = None) -> list[dict]:
    """AI voices from MLAAD: many recent TTS / voice-cloning tools (needs HF_TOKEN).

    lang: 'en' (143 tools incl. ElevenLabs, OpenAI, Cartesia, MiniMax, F5, XTTS...)
          or 'hi' (9 tools incl. IndicF5, Veena, XTTS-v2).
    Tools named in `holdout` are never trained on: they become the
    "new tools" test. Other tools: ~90% of files train, ~10% dev.
    """
    from concurrent.futures import ThreadPoolExecutor
    from huggingface_hub import HfApi, hf_hub_download
    repo = "mueller91/MLAAD"
    api = HfApi(token=token)
    tools = sorted(t.path.split("/")[-1] for t in api.list_repo_tree(repo, f"fake/{lang}", repo_type="dataset")
                   if not t.path.endswith((".csv", ".txt", ".json")))
    if max_tools:
        keep = [t for t in tools if t in holdout]
        rest = [t for t in tools if t not in holdout]
        tools = keep + rest[:: max(1, len(rest) // max_tools)][:max_tools]
    out = Path(out_dir) / f"mlaad_{lang}"
    out.mkdir(parents=True, exist_ok=True)

    jobs = []
    for tool in tools:
        files = sorted(f.path for f in api.list_repo_tree(repo, f"fake/{lang}/{tool}", repo_type="dataset")
                       if f.path.endswith((".wav", ".mp3", ".flac")))
        n = per_holdout_tool if (tool in holdout and per_holdout_tool) else per_tool
        step = max(1, len(files) // n)
        jobs += [(tool, f) for f in files[::step][:n]]
    print(f"  MLAAD {lang}: {len(tools)} tools, {len(jobs)} files to download")

    def fetch(job):
        tool, f = job
        x = None
        for attempt in range(3):                  # retry: many small downloads can hit rate limits
            try:
                p = hf_hub_download(repo, f, repo_type="dataset", token=token, local_dir=str(out / "_download"))
                x = audio.load_audio(p)[: 8 * audio.SR]
                Path(p).unlink(missing_ok=True)
                break
            except Exception:
                time.sleep(2 + 4 * attempt)
        if x is None:
            return None
        if len(x) < audio.SR:
            return None
        dst = out / f"{zlib.crc32(f.encode()):08x}.wav"
        sf.write(dst, x, audio.SR, subtype="PCM_16")
        split = "test" if tool in holdout else _speaker_split(f)      # split by file within a tool
        return {"path": str(dst), "label": "fake", "lang": lang, "source": "mlaad",
                "speaker": f"mlaad_{tool}", "generator": tool, "split": split}

    with ThreadPoolExecutor(workers) as ex:
        rows = [r for r in ex.map(fetch, jobs) if r]
    print(f"  MLAAD {lang}: {len(rows)} clips ({sum(r['split'] == 'test' for r in rows)} from held-out tools)")
    return rows


def asvspoof2019_la(search_root: str = "/kaggle/input") -> list[dict]:
    """English real + fake from an attached ASVspoof 2019 LA dataset (train and dev)."""
    rows = []
    for part, proto_name, split in [("train", "ASVspoof2019.LA.cm.train.trn.txt", "train"),
                                    ("dev", "ASVspoof2019.LA.cm.dev.trl.txt", "dev")]:
        proto = find_file(proto_name, search_root)
        if not proto:
            raise FileNotFoundError(f"{proto_name} not found. Attach the ASVspoof 2019 DATASET "
                                    f"(not a notebook) that has the LA folder.")
        la_root = os.path.dirname(os.path.dirname(proto))          # .../LA
        flac = os.path.join(la_root, f"ASVspoof2019_LA_{part}", "flac")
        if not os.path.isdir(flac):
            # some uploads nest the folder once more, e.g. ASVspoof2019_LA_train/ASVspoof2019_LA_train/flac
            hits = [d for d, dirs, _ in os.walk(la_root) if d.endswith(os.path.join(f"ASVspoof2019_LA_{part}", "flac"))]
            if not hits:
                raise FileNotFoundError(f"No ASVspoof2019_LA_{part}/flac folder under {la_root}.")
            flac = hits[0]
        for line in open(proto):
            spk, key, _, system, label = line.split()
            rows.append({"path": f"{flac}/{key}.flac", "label": "real" if label == "bonafide" else "fake",
                         "lang": "en", "source": "asvspoof19", "speaker": spk,
                         "generator": "" if system == "-" else system, "split": split})
    return rows


def in_the_wild(search_root: str = "/kaggle/input") -> list[dict]:
    """English real-world deepfakes. Test only, never trained on."""
    meta = find_file("meta.csv", search_root)
    if not meta:
        return []
    df = pd.read_csv(meta)
    base = os.path.dirname(meta)
    return [{"path": f"{base}/{r.file}", "label": "fake" if str(r.label).lower() == "spoof" else "real",
             "lang": "en", "source": "in_the_wild", "speaker": f"itw_{r.speaker}", "generator": "",
             "split": "test"} for r in df.itertuples()]


def extra_labels(search_root: str = "/kaggle/input") -> list[dict]:
    """Optional: our own recordings, from a dataset that has vs_labels.csv.

    Columns: file,label,language,speaker,split[,tool]
      label     real | fake
      language  en | hi | hinglish   (Hinglish is grouped with Hindi)
      speaker   the PERSON (for a clone: the person whose voice was cloned)
      split     train | test   (the same person must never appear in both)
      tool      optional, e.g. xtts, elevenlabs, f5
    """
    found = find_file("vs_labels.csv", search_root)
    if not found:
        return []
    df = pd.read_csv(found)
    base = os.path.dirname(found)
    clash = set(df[df.split == "train"].speaker) & set(df[df.split == "test"].speaker)
    if clash:
        raise ValueError(f"These people are in both train and test, which makes the test meaningless: {sorted(clash)}")
    rows = []
    for r in df.itertuples():
        lang = "hi" if str(r.language).lower().startswith(("hi", "hinglish")) else "en"
        tool = getattr(r, "tool", "")
        rows.append({"path": f"{base}/{r.file}", "label": r.label, "lang": lang, "source": "ours",
                     "speaker": f"ours_{r.speaker}", "generator": str(tool) if pd.notna(tool) else "",
                     "split": r.split})
    return rows


def summary(rows: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    return df.groupby(["split", "lang", "label", "source"]).size().unstack("label", fill_value=0)


# ---------------------------------------------------------------- torch datasets

class TrainSet(Dataset):
    def __init__(self, rows: list[dict]):
        self.rows = rows
        self.rng = np.random.default_rng()

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        try:
            x = audio.load_audio(r["path"])
        except Exception:
            x = np.zeros(audio.SR, np.float32)
        x = augment(x, self.rng)
        return torch.from_numpy(x), 1 if r["label"] == "real" else 0    # model index 1 = real


class EvalSet(Dataset):
    def __init__(self, rows: list[dict], phone: bool):
        self.rows, self.phone = rows, phone

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        try:
            x = audio.load_audio(self.rows[i]["path"])
        except Exception:
            x = np.zeros(audio.SR, np.float32)
        return torch.from_numpy(eval_view(x, self.phone, seed=i)), i


def worker_init(worker_id: int):
    info = torch.utils.data.get_worker_info()
    info.dataset.rng = np.random.default_rng(torch.initial_seed() % 2**32 + worker_id)


def _bucket(r: dict) -> str:
    """What gets an equal share inside a group: the AI tool for fakes, the dataset for real voices."""
    if r["source"] == "ours":
        return "ours"
    if r["label"] == "fake":
        gen = r.get("generator") or r["source"]
        return "xtts_v2" if "xtts_v2" in gen else gen      # same tool, named differently by two datasets
    return r["source"]


def balanced_weights(rows: list[dict], ours_share: float = 0.05) -> torch.Tensor:
    """Sampling weights for balanced batches.

    1. The 4 groups English-real, English-AI, Hindi-real, Hindi-AI are drawn
       equally often, so overall it is 50% real and 50% AI, 50% English and 50% Hindi.
    2. Inside an AI group every TOOL gets the same share (ElevenLabs, XTTS-v2,
       FreeVC, each ASVspoof system...), however many clips it has.
    3. Inside a real group every dataset gets the same share.
    4. Our own recordings get a fixed `ours_share` of each group, for real and AI
       alike: enough to matter, not so much that a few dozen clips are memorized.
    """
    from collections import Counter
    key = [(r["lang"], r["label"], _bucket(r)) for r in rows]
    per_bucket = Counter(key)
    others_in_group = Counter(k[:2] for k in per_bucket if k[2] != "ours")
    has_ours = {k[:2] for k in per_bucket if k[2] == "ours"}
    w = []
    for k in key:
        g = k[:2]
        if k[2] == "ours":
            share = ours_share if others_in_group[g] else 1.0
        else:
            share = (1.0 - (ours_share if g in has_ours else 0.0)) / others_in_group[g]
        w.append(share / per_bucket[k])
    return torch.tensor(w, dtype=torch.double)


def mix_report(rows: list[dict]) -> pd.DataFrame:
    """Share of training draws per group and per tool/dataset, as the sampler will see them."""
    w = balanced_weights(rows).numpy()
    df = pd.DataFrame({"lang": [r["lang"] for r in rows], "label": [r["label"] for r in rows],
                       "bucket": [_bucket(r) for r in rows], "w": w / w.sum()})
    t = df.groupby(["lang", "label", "bucket"]).agg(clips=("w", "size"), share=("w", "sum")).reset_index()
    t["share %"] = (t.pop("share") * 100).round(2)
    return t


In [ ]:
%%writefile vs/train.py
"""Fine-tuning loop and evaluation for the SLS detector."""

from __future__ import annotations

import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler

from .metrics import compute_eer
from .train_data import EvalSet, TrainSet, balanced_weights, worker_init


@torch.inference_mode()
def predict(model, rows: list[dict], device: str, phone: bool, batch: int = 32,
            workers: int = 4) -> np.ndarray:
    """Fake-probability for each row."""
    was_training = model.training
    model.eval()
    p = np.zeros(len(rows))
    dl = DataLoader(EvalSet(rows, phone), batch_size=batch, num_workers=workers)
    with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
        for x, idx in dl:
            p[idx.numpy()] = 1 - model(x.to(device)).float()[:, 1].exp().cpu().numpy()
    model.train(was_training)
    return p


def evaluate(model, groups: dict[str, list[dict]], device: str, workers: int = 4) -> pd.DataFrame:
    """EER and false alarms for each named group, clean and phone mode."""
    out = []
    for name, rows in groups.items():
        y = np.array([r["label"] == "fake" for r in rows])
        for phone in (False, True):
            p = predict(model, rows, device, phone, workers=workers)
            row = {"group": name, "mode": "phone" if phone else "clean",
                   "real": int((~y).sum()), "fake": int(y.sum())}
            if y.any() and (~y).any():
                eer, thr = compute_eer(p[~y], p[y])
                row.update({"EER %": round(eer * 100, 2), "threshold": round(thr, 3)})
            if (~y).any():
                row["false alarms on real @0.5 %"] = round(float(np.mean(p[~y] > 0.5)) * 100, 1)
            if y.any():
                row["missed fakes @0.5 %"] = round(float(np.mean(p[y] <= 0.5)) * 100, 1)
            out.append(row)
    return pd.DataFrame(out)


def selection_score(table: pd.DataFrame, select_groups: list[str]) -> float:
    """Lower is better: mean EER over the dev groups, clean and phone."""
    t = table[table["group"].isin(select_groups)]
    return float(t["EER %"].mean())


def finetune(model, train_rows: list[dict], dev_groups: dict[str, list[dict]], *, device: str,
             out_path: str, steps: int = 2000, batch: int = 8, accum: int = 2,
             lr_ssl: float = 2e-6, lr_head: float = 5e-5, warmup: int = 150,
             eval_every: int = 400, workers: int = 4, log=print) -> list[dict]:
    """Fine-tune and keep the checkpoint with the best dev EER.

    steps counts optimizer updates; each update sees batch * accum clips.
    """
    model.ssl.freeze_feature_encoder()              # the CNN front end stays as pretrained
    head = [p for n, p in model.named_parameters() if not n.startswith("ssl.") and p.requires_grad]
    ssl = [p for n, p in model.named_parameters() if n.startswith("ssl.") and p.requires_grad]
    opt = torch.optim.AdamW([{"params": ssl, "lr": lr_ssl}, {"params": head, "lr": lr_head}],
                            weight_decay=1e-4)
    base = [lr_ssl, lr_head]
    scaler = torch.amp.GradScaler(enabled=device == "cuda")

    sampler = WeightedRandomSampler(balanced_weights(train_rows), num_samples=steps * batch * accum,
                                    replacement=True)
    dl = DataLoader(TrainSet(train_rows), batch_size=batch, sampler=sampler, num_workers=workers,
                    worker_init_fn=worker_init, drop_last=True, persistent_workers=workers > 0)

    history, best = [], math.inf
    select = list(dev_groups)
    model.train()
    t0, step, micro, run_loss = time.time(), 0, 0, 0.0
    for x, y in dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
            loss = F.nll_loss(model(x).float(), y) / accum
        scaler.scale(loss).backward()
        run_loss += loss.item()
        micro += 1
        if micro % accum:
            continue

        step += 1
        scale = min(1.0, step / warmup) * 0.5 * (1 + math.cos(math.pi * min(step / steps, 1.0)))
        for g, b in zip(opt.param_groups, base):
            g["lr"] = b * max(scale, 0.02)
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)

        if step % 50 == 0:
            log(f"step {step}/{steps}  loss {run_loss / 50:.4f}  {(time.time() - t0) / 60:.1f} min")
            run_loss = 0.0
        if step % eval_every == 0 or step == steps:
            table = evaluate(model, dev_groups, device, workers)
            score = selection_score(table, select)
            history.append({"step": step, "dev_mean_eer": round(score, 2),
                            "minutes": round((time.time() - t0) / 60, 1)})
            log(table.to_string(index=False))
            # save on the first evaluation no matter what, so the next cell always has a file
            if score < best or not Path(out_path).exists():
                best = score if not math.isnan(score) else best
                save(model, out_path)
                log(f"  new best dev EER {score:.2f}% -> saved {out_path}")
            else:
                log(f"  dev EER {score:.2f}% (best {best:.2f}%)")
            model.train()
        if step >= steps:
            break
    Path(out_path).with_suffix(".history.json").write_text(json.dumps(history, indent=1))
    return history


def save(model, path: str) -> None:
    """Half-precision state dict (~680 MB). vs.model.load_sls loads it directly."""
    sd = {k: (v.half() if v.is_floating_point() else v) for k, v in model.state_dict().items()}
    torch.save(sd, path)


In [ ]:
QUICK_TEST = False       # True = tiny dry run (~15 min) to check the setup
RUN = "A"                # "A" = keep the 10 newest tools out (proves generalization), start from the public model
                         # "B" = final model: train on every tool, continue from run A's model
                         #       (attach run A's output voice_shield_ft.pth as an input first)

STEPS, BATCH, ACCUM, EVAL_EVERY = 3000, 8, 2, 500
# Hindi IndicSynth shards are sorted by tool: about 0-54 FreeVC, 56-106 XTTS-v2. Take 6 of each.
INDICSYNTH_SHARDS = [0, 9, 18, 27, 36, 45, 56, 66, 76, 86, 96, 106]
PER_SHARD = 700
HOLDOUT_XTTS = False     # True = never train on XTTS-v2, report it as an unseen tool
FLEURS_TRAIN, FLEURS_EVAL = 2500, 400        # real speech per language
KATHBATH_SHARDS, KB_PER_SHARD = [0, 8, 16, 24], 800
SVARAH_TRAIN, SVARAH_TEST, SVARAH_PER_SHARD = [0, 1], [2], 1500   # Indian-accented English, 3 files in total
ASV_TRAIN_FAKE = 6000                         # all ~2.6k real ASVspoof clips are used
EVAL_PER_CLASS, ITW_PER_CLASS = 400, 1500
LIBRI = {"train.clean.100": [0, 7], "train.other.500": [0, 32], "test.clean": [0]}
LIBRI_PER_SHARD = 700
MLAAD_EN_PER_TOOL, MLAAD_HI_PER_TOOL, MLAAD_HOLDOUT_PER_TOOL = 40, 300, 60   # clips per tool
MLAAD_MAX_EN_TOOLS = None                     # None = every English tool
# Newest tools kept out of training -> "never trained on" test rows. Set to [] for the final model.
MLAAD_HOLDOUT_EN = ["ElevenLabs-v3", "OpenAI TTS-1 HD", "Cartesia.ai (Sonic-3)", "Gemini-3.1-Flash-TTS",
                    "MiniMax-Speech-2.8-Turbo", "Resemble.ai (April 12th, 2025)", "Inworld-TTS-2", "Chatterbox-Turbo"]
MLAAD_HOLDOUT_HI = ["Veena", "Ringg Squirrel TTS v1.0"]
DATA = "/tmp/vs_data"

if RUN == "B":
    MLAAD_HOLDOUT_EN, MLAAD_HOLDOUT_HI = [], []   # nothing held out: the demo model sees every tool
    STEPS, EVAL_EVERY = 1500, 500

if QUICK_TEST:
    STEPS, EVAL_EVERY = 40, 20
    INDICSYNTH_SHARDS, PER_SHARD = [0, 60], 150
    FLEURS_TRAIN, FLEURS_EVAL = 150, 60
    KATHBATH_SHARDS, KB_PER_SHARD = [0], 100
    SVARAH_TRAIN, SVARAH_TEST, SVARAH_PER_SHARD = [0], [2], 60
    ASV_TRAIN_FAKE, EVAL_PER_CLASS, ITW_PER_CLASS = 300, 60, 100
    LIBRI, LIBRI_PER_SHARD = {"train.clean.100": [0], "test.clean": [0]}, 100
    MLAAD_EN_PER_TOOL, MLAAD_HI_PER_TOOL, MLAAD_HOLDOUT_PER_TOOL, MLAAD_MAX_EN_TOOLS = 3, 10, 10, 12

In [ ]:
import sys, glob, os, random, shutil
sys.path.insert(0, os.getcwd())     # the vs/ package written by the cells above

import numpy as np, pandas as pd, torch
from vs import train_data as td
from vs.model import load_sls, download_checkpoint
from vs.train import evaluate, finetune

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "(no GPU: turn it on in Settings)")
from vs import audio as _a
print("ffmpeg:", shutil.which("ffmpeg"), "| AMR/GSM phone codecs:",
      "available" if _a._ffmpeg_codec(np.zeros(8000, "float32"), "amr") is not None else
      "not available (phone simulation uses telephone band + G.711 only)")
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None
print("HuggingFace token:", "found" if HF_TOKEN else
      "NOT SET: MLAAD (newest AI voices), Svarah and Kathbath will be skipped. See step 5 at the top.")
START = None
if RUN == "B":
    START = td.find_file("voice_shield_ft.pth")
    assert START, ("Run B needs run A's model: Add Input → Your Work → Notebooks → pick the run A notebook "
                   "(its output has voice_shield_ft.pth).")
print("Run", RUN, "| starting from:", START or "public XLS-R + SLS checkpoint")

## 1 · Collect data (about 15–30 minutes the first time)

In [ ]:
random.seed(0)
rows = []

asv = td.asvspoof2019_la()
asv_train_real = [r for r in asv if r["split"] == "train" and r["label"] == "real"]
asv_train_fake = random.sample([r for r in asv if r["split"] == "train" and r["label"] == "fake"], ASV_TRAIN_FAKE)
rows += asv_train_real + asv_train_fake + [r for r in asv if r["split"] == "dev"]
print("ASVspoof:", len(rows))

for lang in ["hi_in", "en_us"]:
    rows += td.fleurs(lang, "train", DATA, FLEURS_TRAIN)
    rows += td.fleurs(lang, "validation", DATA, FLEURS_EVAL)
    rows += td.fleurs(lang, "test", DATA, FLEURS_EVAL)
print("+ FLEURS:", len(rows))
rows += td.librispeech(LIBRI, DATA, LIBRI_PER_SHARD)
print("+ LibriSpeech:", len(rows))

rows += td.indicsynth_hindi(INDICSYNTH_SHARDS, DATA, PER_SHARD, holdout_xtts=HOLDOUT_XTTS)
if HF_TOKEN:
    try:
        rows += td.kathbath_hindi(KATHBATH_SHARDS, DATA, KB_PER_SHARD, HF_TOKEN)
    except Exception as e:
        print("Kathbath skipped:", e)
    try:
        rows += td.svarah(SVARAH_TRAIN, SVARAH_TEST, DATA, SVARAH_PER_SHARD, HF_TOKEN)
    except Exception as e:
        print("Svarah skipped (did you accept its terms on huggingface.co?):", e)
    for lang, per_tool, holdout, max_tools in [("en", MLAAD_EN_PER_TOOL, MLAAD_HOLDOUT_EN, MLAAD_MAX_EN_TOOLS),
                                               ("hi", MLAAD_HI_PER_TOOL, MLAAD_HOLDOUT_HI, None)]:
        try:
            rows += td.mlaad(lang, DATA, per_tool, HF_TOKEN, holdout, max_tools=max_tools,
                             per_holdout_tool=MLAAD_HOLDOUT_PER_TOOL)
        except Exception as e:
            print(f"MLAAD {lang} skipped (did you accept its terms on huggingface.co?):", e)
else:
    print("No HF_TOKEN: skipping Kathbath, Svarah and MLAAD (newest AI voices).")
itw = td.in_the_wild()
rows += td.extra_labels()
print("Total clips:", len(rows), "| In-the-Wild available:", len(itw))
display(td.summary(rows))
du = shutil.disk_usage("/tmp")
print(f"Disk: {du.used / 1e9:.1f} GB used, {du.free / 1e9:.1f} GB free")
print(pd.Series([r["generator"] for r in rows if r["source"] == "indicsynth"]).value_counts())
for lang in ["en", "hi"]:
    m = [r for r in rows if r["source"] == "mlaad" and r["lang"] == lang]
    print(f"MLAAD {lang}: {len({r['generator'] for r in m})} tools, {len(m)} clips, "
          f"{sum(r['split'] == 'test' for r in m)} from held-out tools")

In [ ]:
def pick(rs, n, seed=0):
    rs = list(rs); random.Random(seed).shuffle(rs); return rs[:n]

def by(**kw):
    return [r for r in rows if all(r[k] == v for k, v in kw.items())]

def pick_each(rs, n, key="source"):
    # n clips spread equally over sources (or tools), so one big dataset can't fill a test group
    groups = {}
    for r in rs:
        groups.setdefault(r[key] if key != "tool" else (r["generator"] or r["source"]), []).append(r)
    out = []
    for i, k in enumerate(sorted(groups)):
        out += pick(groups[k], max(1, n // max(len(groups), 1)), seed=i)
    return out

train_rows = [r for r in rows if r["split"] == "train"]

half = EVAL_PER_CLASS // 2
dev_groups = {
    # English fakes: half older tools (ASVspoof), half newer tools (MLAAD) when available
    "English dev": pick_each(by(split="dev", lang="en", label="real"), EVAL_PER_CLASS * 2)
                   + pick(by(split="dev", source="asvspoof19", label="fake"), half)
                   + pick(by(split="dev", source="mlaad", lang="en"), half),
    "Hindi dev":   pick_each(by(split="dev", lang="hi", label="real"), EVAL_PER_CLASS)
                   + pick_each(by(split="dev", lang="hi", label="fake"), EVAL_PER_CLASS, key="tool"),
}
en_test_real = pick_each(by(split="test", lang="en", label="real"), EVAL_PER_CLASS)   # FLEURS + LibriSpeech + Svarah
test_groups = {
    "English real voices (FLEURS + LibriSpeech + Svarah test)": en_test_real,
    "Hindi real voices (FLEURS test)": pick(by(split="test", source="fleurs_hi"), EVAL_PER_CLASS),
}
if by(split="test", source="svarah"):
    test_groups["Indian-accented English real voices (Svarah test)"] = pick(by(split="test", source="svarah"), EVAL_PER_CLASS)
if by(split="test", source="mlaad", lang="en"):
    test_groups["English, newest AI tools never trained on"] = (
        en_test_real + pick_each(by(split="test", source="mlaad", lang="en"), EVAL_PER_CLASS, key="tool"))
if by(split="test", source="mlaad", lang="hi"):
    test_groups["Hindi, newest AI tools never trained on"] = (
        pick(by(split="test", source="fleurs_hi"), EVAL_PER_CLASS) + pick(by(split="test", source="mlaad", lang="hi"), EVAL_PER_CLASS))
if HOLDOUT_XTTS:
    test_groups["Hindi, unseen tool (XTTS-v2)"] = (pick(by(split="test", source="fleurs_hi"), EVAL_PER_CLASS)
                                                  + pick(by(split="test", generator="xtts_v2"), EVAL_PER_CLASS))
if itw:
    test_groups["English real world (In-the-Wild)"] = (pick([r for r in itw if r["label"] == "real"], ITW_PER_CLASS)
                                                      + pick([r for r in itw if r["label"] == "fake"], ITW_PER_CLASS))
ours = [r for r in rows if r["source"] == "ours" and r["split"] == "test"]
if ours:
    test_groups["Our recordings (all, 8 kHz)"] = ours
    test_groups["Our recordings, English"] = [r for r in ours if r["lang"] == "en"]
    test_groups["Our recordings, Hindi + Hinglish"] = [r for r in ours if r["lang"] == "hi"]
for name, g in {**dev_groups, **test_groups}.items():
    print(f"{name:40s} real {sum(r['label']=='real' for r in g):5d}  fake {sum(r['label']=='fake' for r in g):5d}")
print("Training clips:", len(train_rows))

# How training batches will be mixed: 25% per group, and an equal share per AI tool / real dataset inside each group
mix = td.mix_report(train_rows)
print(mix.groupby(["lang", "label"])[["clips", "share %"]].sum())
pd.set_option("display.max_rows", 300)
display(mix)

## 2 · Before: the starting model (ready-made for run A, run A's model for run B)

In [ ]:
model = load_sls(START or download_checkpoint(), device)
before = evaluate(model, {**dev_groups, **test_groups}, device)
before.to_csv("/kaggle/working/before.csv", index=False)
before

## 3 · Fine-tune

In [ ]:
OUT = "/kaggle/working/voice_shield_ft.pth" if RUN == "A" else "/kaggle/working/voice_shield_final.pth"
history = finetune(model, train_rows, dev_groups, device=device, out_path=OUT,
                   steps=STEPS, batch=BATCH, accum=ACCUM, eval_every=EVAL_EVERY)
pd.DataFrame(history)

## 4 · After: our fine-tuned model, and the comparison

In [ ]:
del model; torch.cuda.empty_cache()
model = load_sls(OUT, device)
after = evaluate(model, {**dev_groups, **test_groups}, device)
after.to_csv("/kaggle/working/after.csv", index=False)

cols = ["EER %", "false alarms on real @0.5 %", "missed fakes @0.5 %"]
cmp = before[["group", "mode"] + [c for c in cols if c in before]].merge(
      after[["group", "mode"] + [c for c in cols if c in after]], on=["group", "mode"], suffixes=(" before", " after"))
cmp.to_csv("/kaggle/working/before_vs_after.csv", index=False)
print("Copy this table into the chat:")
print(cmp.to_string(index=False))